In [0]:
# Programmatically restart python interpreter to load pyx12 2.3.3
dbutils.library.restartPython()
%pip install pyx12==4.0.0
# Databricks notebook source
# DBTITLE 1,Install Required Libraries
%pip install recordlinkage -q
dbutils.widgets.removeAll()
%pip install pyedi

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import os
import sys
import shutil
from pathlib import Path
import json

dbutils.widgets.removeAll()

# Force local file system synchronization
os.sync()

# Absolute workspace configuration matching colleague working setup
ROOT_DIR = Path("/Workspace/Users/saythu000@gmail.com/claimprocessing")
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

In [0]:
from shared.EDIProcessing import EDIProcessor, CSVConverter
from DimMember.EDIProcessing.mapper import Mapper

In [0]:
def move_file(src_path: Path, target_dir: Path) -> Path:
    """Moves a file to a target directory cleanly, ensuring the directory exists."""
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / src_path.name
    shutil.move(str(src_path), str(target_path))
    return target_path

def finalize_file_tracking(file_path: Path, base_source_dir: Path, success: bool):
    """Moves the raw EDI file out of inprogress based on lifecycle completion status."""
    try:
        if success:
            print(f"--> Cycle Complete. Archiving raw file to processed: {file_path.name}")
            move_file(file_path, base_source_dir / "processed")
        else:
            print(f"--> Cycle Failed. Moving raw file to failed triage: {file_path.name}")
            move_file(file_path, base_source_dir / "failed")
    except Exception as e:
        print(f"Failed to cleanly update tracking directory state: {e}")

In [0]:
def process_single_file(incoming_file_path: Path, base_source_dir: Path) -> tuple:
    """Parses an EDI file, extracts metadata, maps records, and generates a targeted CSV."""
    active_file_path = incoming_file_path
    try:
        if not active_file_path.exists():
            raise FileNotFoundError(f"Input file missing: {active_file_path}")
        
        active_file_path = move_file(active_file_path, base_source_dir / "inprogress")

        # Core Parsing & Domain Mapping
        structured_json = EDIProcessor().parse(str(active_file_path))
        
        # Metadata Extraction
        interchange = structured_json.get('interchange', {})
        client_id = interchange.get('sender_id', '').strip()
        file_id = interchange.get('control_number', '').strip()
        
        st_segment = structured_json.get('heading', {}).get('transaction_set_header_loop', {}).get('transaction_set_header_ST', {})
        layout_id = st_segment.get('transaction_set_identifier_code', '834').strip()
        
        # Isolated CSV Delivery Production Rules
        target_csv_name = f"{active_file_path.stem}.csv"
        target_csv_path = ROOT_DIR / "temp" / layout_id / target_csv_name
        target_csv_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Code Conversion Execution
        CSVConverter().converter(Mapper().map_member(structured_json), str(target_csv_path))
             
        print("client id: ", client_id, " file id: ", file_id, " layout id: ", layout_id, " csv name: ", target_csv_name)
        return client_id, file_id, layout_id, target_csv_name, active_file_path
        
    except Exception as e:
        print(f"Failed processing {active_file_path.name}: {e}")
        if active_file_path.exists():
            move_file(active_file_path, base_source_dir / "failed")
        raise

In [0]:
def build_payloads(processed_files: list) -> tuple:
    """Generates precise production payloads for downstream orchestration stages."""
    process_list = []
    consolidation_list = []
    
    for f in processed_files:
        specific_client_container = f"{ROOT_DIR}/temp/{f['layout_id']}"
        
        process_list.append({
            "ClientID": f['client_id'],
            "FileID": f['file_id'],
            "FileName": f['csv_filename'],
            "ClientContainer": specific_client_container,
            "CurrentFolderPath": "",
            "ProcessedFolderPath": "/Volumes/claimsprocessing/bronze/member",
            "ColumnDelimiter": ",",
            "HasHeader": "true",
            "IgnoreHeader": "False",
            "FileLayoutID": f['layout_id'],
            "FileLayoutDescription": f"Standard{f['layout_id']}",
            "SchemaFileName": "MemberSchema.json",
            "SchemaFilePath": f"{ROOT_DIR}/DimMember/Bronze/Schema",
            "TextQualifier": "\""
        })
        
        consolidation_list.append({
            "DataGroupTrackingID": f"TRACK_MEMBER_{f['layout_id']}_{f['file_id']}",
            "DataGroupMappingId": f"{f['layout_id']}MEMBER",
            "FileId": f['file_id'],
            "FileLayoutID": f['layout_id'],
            "FileLayoutDescription": f"Standard{f['layout_id']}",
            "CurrentContainer": "/Volumes/claimsprocessing/bronze/member",
            "CurrentFolderPath": "",
            "ConsolidatedMappingFilePath": f"{ROOT_DIR}/DimMember/Bronze/Schema/Consolidation",
            "ConsolidatedMappingFileName": "ConsolidationMember.json",
            "ConsolidatedLayerDataModelFilePath": f"{ROOT_DIR}/DimMember/Bronze/Schema/Consolidation/DataModels",
            "ConsolidatedLayerDataModel": "MemberDataModel.json",
            "ConsolidatedFolderPath": "/Volumes/claimsprocessing/bronze/member_consolidated"
        })
        
    return json.dumps({"FileIds": process_list}), json.dumps({"FileIds": consolidation_list})

def trigger_silver_notebooks():
    """Triggers Silver layer notebooks in dependency order."""
    silver_notebooks_base = f"{ROOT_DIR}/DimMember/Silver/Notebooks"
    
    try:
        print("\n=== Triggering MemberPersonBridge (Silver Layer) ===")
        dbutils.notebook.run(f"{silver_notebooks_base}/MemberPersonBridge", 600)
        print("MemberPersonBridge completed successfully")
        
        print("\n=== Triggering Member (Silver Layer) ===")
        dbutils.notebook.run(f"{silver_notebooks_base}/Member", 600)
        print("Member completed successfully")
        
        print("\n=== Triggering MemberGroup (Silver Layer for gold_ma_membergroup) ===")
        dbutils.notebook.run(f"{silver_notebooks_base}/MemberGroup", 600)
        print("MemberGroup completed successfully")
        
        print("\nAll Silver layer notebooks completed successfully.")
    except Exception as e:
        print(f"Silver layer processing failed: {e}")
        raise

def build_gold_payload(processed_files: list) -> str:
    """Generates Gold layer payload for GenericSubGroupProcessing."""
    if not processed_files:
        return json.dumps({"FileIds": []})
    
    first_file = processed_files[0]
    gold_entry = {
        "FileId": first_file['file_id'],
        "ClientID": first_file['client_id'],
        "SourceContainer": "claimsprocessing.silver.member",
        "FileLayoutID": first_file['layout_id'],
        "FileLayoutDescription": f"Standard{first_file['layout_id']}"
    }
    return json.dumps({"FileIds": [gold_entry]})

def trigger_gold_notebooks(gold_payload: str):
    """Triggers Gold layer notebook for dimension processing."""
    gold_notebooks_base = f"{ROOT_DIR}/DimMember/Gold/Notebooks"
    
    try:
        print("\n=== Triggering GenericSubGroupProcessing (Gold Layer) ===")
        dbutils.notebook.run(
            f"{gold_notebooks_base}/GenericSubGroupProcessing", 
            600, 
            {"GoldProcessingJSON": gold_payload}
        )
        print("GenericSubGroupProcessing completed successfully")
        print("\nAll Gold layer notebooks completed successfully.")
        
    except Exception as e:
        print(f"Gold layer processing failed: {e}")
        raise

In [0]:
def main():
    base_source_dir = ROOT_DIR / "source/834"
    pending_dir = base_source_dir / "pending"
    
    if not pending_dir.exists():
        print(f"Pending directory does not exist: {pending_dir}")
        return
        
    incoming_files = [f for f in pending_dir.iterdir() if f.is_file() and not f.name.startswith('.')]
    if not incoming_files:
        print("No files found to process.")
        return

    notebook_base = f"{ROOT_DIR}/shared/Notebooks"

    for file_path in incoming_files:
        active_tracked_file = None
        try:
            c_id, f_id, l_id, csv_filename, active_tracked_file = process_single_file(file_path, base_source_dir)
            
            single_file_list = [{
                'client_id': c_id, 
                'file_id': f_id, 
                'layout_id': l_id,
                'csv_filename': csv_filename
            }]
            
            process_payload, consolidation_payload = build_payloads(single_file_list)
            gold_payload = build_gold_payload(single_file_list)
            
            print(f"\n=======================================================")
            print(f"STARTING LIFECYCLE FOR DETACHED FILE: {csv_filename}")
            print(f"=======================================================")

            print("\n=== Triggering FilesToProcess (Bronze) ===")
            dbutils.notebook.run(f"{notebook_base}/FilesToProcess", 600, {"ProcessedJSON": process_payload})
            
            print("\n=== Triggering LoopConsolidationFiles (Bronze Consolidated) ===")
            dbutils.notebook.run(f"{notebook_base}/LoopConsolidationFiles", 600, {"ConsolidationJSON": consolidation_payload})
            
            trigger_silver_notebooks()
            trigger_gold_notebooks(gold_payload)
            
            finalize_file_tracking(active_tracked_file, base_source_dir, success=True)
            print(f"\nFull end-to-end processing successful for file: {csv_filename}\n")
            
        except Exception as e:
            print(f"Pipeline crashed during track processing: {e}")
            target_to_fail = active_tracked_file if active_tracked_file else file_path
            finalize_file_tracking(target_to_fail, base_source_dir, success=False)
            continue

if __name__ == "__main__":
    main()

Error at line 4: 'walk_tree' object has no attribute 'walk'
Error at line 5: 'walk_tree' object has no attribute 'walk'
Error at line 6: 'walk_tree' object has no attribute 'walk'
Error at line 7: 'walk_tree' object has no attribute 'walk'
Error at line 8: 'walk_tree' object has no attribute 'walk'
Error at line 9: 'walk_tree' object has no attribute 'walk'
Error at line 10: 'walk_tree' object has no attribute 'walk'
Error at line 11: 'walk_tree' object has no attribute 'walk'
Error at line 12: 'walk_tree' object has no attribute 'walk'
Error at line 13: 'walk_tree' object has no attribute 'walk'
Error at line 14: 'walk_tree' object has no attribute 'walk'
Error at line 15: 'walk_tree' object has no attribute 'walk'
Error at line 16: 'walk_tree' object has no attribute 'walk'
Error at line 17: 'walk_tree' object has no attribute 'walk'
Error at line 18: 'walk_tree' object has no attribute 'walk'
Error at line 19: 'walk_tree' object has no attribute 'walk'
Error at line 20: 'walk_tree' 

client id:  CITY834N  file id:  900000004  layout id:  834  csv name:  stuart.csv

STARTING LIFECYCLE FOR DETACHED FILE: stuart.csv

=== Triggering FilesToProcess (Bronze) ===

=== Triggering LoopConsolidationFiles (Bronze Consolidated) ===

=== Triggering MemberPersonBridge (Silver Layer) ===
MemberPersonBridge completed successfully

=== Triggering Member (Silver Layer) ===
Member completed successfully

=== Triggering MemberGroup (Silver Layer for gold_ma_membergroup) ===
MemberGroup completed successfully

All Silver layer notebooks completed successfully.

=== Triggering GenericSubGroupProcessing (Gold Layer) ===
GenericSubGroupProcessing completed successfully

All Gold layer notebooks completed successfully.
--> Cycle Complete. Archiving raw file to processed: stuart.txt

Full end-to-end processing successful for file: stuart.csv



In [0]:
#step-1 debuging the for the csv file generation

from pathlib import Path
from shared.EDIProcessing import EDIProcessor, CSVConverter
from DimMember.EDIProcessing.mapper import Mapper

ROOT_DIR = Path("/Workspace/Users/saythu000@gmail.com/claimprocessing")
file_path = ROOT_DIR / "source/834/pending/stuart.txt"

# 1. Parse EDI 834
structured_json = EDIProcessor().parse(str(file_path))
print("1. Structured JSON parsed:", list(structured_json.keys()))

# 2. Map Member Data
mapped_data = Mapper().map_member(structured_json)
print("2. Mapped Data:", mapped_data)

# 3. Write CSV to Workspace
target_csv_path = ROOT_DIR / "temp/834/stuart.csv"
target_csv_path.parent.mkdir(parents=True, exist_ok=True)
CSVConverter().converter(mapped_data, str(target_csv_path))

print("4. Does CSV file exist on disk?", target_csv_path.exists())

Error at line 4: 'walk_tree' object has no attribute 'walk'
Error at line 5: 'walk_tree' object has no attribute 'walk'
Error at line 6: 'walk_tree' object has no attribute 'walk'
Error at line 7: 'walk_tree' object has no attribute 'walk'
Error at line 8: 'walk_tree' object has no attribute 'walk'
Error at line 9: 'walk_tree' object has no attribute 'walk'
Error at line 10: 'walk_tree' object has no attribute 'walk'
Error at line 11: 'walk_tree' object has no attribute 'walk'
Error at line 12: 'walk_tree' object has no attribute 'walk'
Error at line 13: 'walk_tree' object has no attribute 'walk'
Error at line 14: 'walk_tree' object has no attribute 'walk'
Error at line 15: 'walk_tree' object has no attribute 'walk'
Error at line 16: 'walk_tree' object has no attribute 'walk'
Error at line 17: 'walk_tree' object has no attribute 'walk'
Error at line 18: 'walk_tree' object has no attribute 'walk'
Error at line 19: 'walk_tree' object has no attribute 'walk'
Error at line 20: 'walk_tree' 

1. Structured JSON parsed: ['interchange', 'functional_group', 'heading', 'detail', 'transaction_type']
2. Mapped Data: {'UNIQUEPERSONKEY': '847362915', 'BENEFICIARYID': '847362915', 'MEDICAIDID': 'M7004918273', 'PLANMEMBERID': 'A7004564738', 'SUBSCRIBERID': 'S7004102938', 'ENROLLEEUNIQUEID': 'B7004556677', 'MASKEDMEMBERID': 'AMERICAN INDIAN OR ALASKA NATIVE', 'ALTERNATEKEY1': '1002-5', 'ALTERNATEKEY2': 'NOT HISPANIC OR LATINO', 'ALTERNATEKEY3': '2186-5', 'ALTERNATEKEY4': 'Y', 'LASTNAME': 'LITTLE', 'FIRSTNAME': 'STUART', 'MIDDLEINITIAL': 'A', 'DATEOFBIRTH': '1992-07-03', 'GENDER': 'M', 'DECEASEDDATE': '2026-08-01', 'PERMANENTADDRESSLINE1': '210 East 73rd Street', 'PERMANENTADDRESSLINE2': 'Apt 12', 'PERMANENTCITY': 'New York', 'PERMANENTSTATE': 'NY', 'PERMANENTZIPCODE': '10022', 'PHONENUMBER': '2125558850', 'EMAIL': 'stuart.little@samplehealth.org', 'SPOKENLANGUAGE': 'ENG', 'OTHERLANGUAGE': 'SPA', 'SUBSCRIBERFLAG': ['Y', 'N'], 'RELATIONSHIPCODE': ['18', '19'], 'MAINTENANCETYPECODE': ['0

In [0]:
print(list(structured_json['detail'].keys()))

['file_effective_date_loop', 'member_level_detail_loop', 'insured_or_subscriber_NM1_loop', 'ref', 'hd', 'dtp', 'ins', 'nm1', 'dmg']


In [0]:
print(structured_json['detail']['insured_or_subscriber_NM1_loop'])

[OrderedDict({'member_name_NM1': [OrderedDict({'entity_identifier_code': 'IL', 'entity_type_qualifier': '1', 'insured_last_name': 'LITTLE', 'insured_first_name': 'STUART', 'insured_middle_name': 'A', 'insured_prefix': None, 'insured_suffix': None, 'insured_id_qualifier': '34', 'insured_id': '847362915'})], 'member_communications_numbers_PER': [OrderedDict({'contact_function_code': 'IP', 'name': None, 'communication_number_qualifier_03': 'HP', 'communication_number_04': '2125558850', 'communication_number_qualifier_05': 'EM', 'communication_number_06': 'stuart.little@samplehealth.org'})], 'member_residence_street_address_N3': OrderedDict({'insured_address_line_1': '210 East 73rd Street', 'insured_address_line_2': 'Apt 12'}), 'member_city_state_zip_code_N4': OrderedDict({'insured_city': 'New York', 'insured_state': 'NY', 'insured_zip_code': '10022', 'insured_country_code': None}), 'member_demographics_DMG': [OrderedDict({'date_time_period_format_qualifier': 'D8', 'prior_incorrect_insured

In [0]:
%sql
SELECT COUNT(*) FROM claimsprocessing.gold.gold_dimmember;
SELECT COUNT(*) FROM claimsprocessing.gold.gold_ma_membergroup;

COUNT(*)
0


In [0]:
df_test = spark.read.option("header", "true").csv("/Workspace/Users/saythu000@gmail.com/claimprocessing/temp/834/stuart.csv")
print("Spark loaded CSV row count:", df_test.count())

Spark loaded CSV row count: 1


In [0]:
from pathlib import Path
schema_path = Path("/Workspace/Users/saythu000@gmail.com/claimprocessing/DimMember/Bronze/Schema/MemberSchema.json")
print("Does MemberSchema.json exist?", schema_path.exists())

Does MemberSchema.json exist? True


In [0]:
from pyspark.sql.functions import lit, current_timestamp

csv_path = "/Workspace/Users/saythu000@gmail.com/claimprocessing/temp/834/stuart.csv"
processed_path = "/Volumes/claimsprocessing/bronze/member"

df_raw_csv = spark.read.format("csv").option("header", "true").load(csv_path)

df_transformed = df_raw_csv \
    .withColumn("FILE_ID", lit(900000004).cast("long")) \
    .withColumn("FILE_LAYOUT_ID", lit(834).cast("int")) \
    .withColumn("FILE_LAYOUT_DESCRIPTION", lit("Standard834")) \
    .withColumn("CLIENT_ID", lit("CITY834N")) \
    .withColumn("LOAD_DATETIME", current_timestamp())

print("Transformed row count:", df_transformed.count())

df_transformed.write.format("parquet").mode("append").save(processed_path)
print("Parquet write completed cleanly!")

Transformed row count: 1
Parquet write completed cleanly!


In [0]:
display(dbutils.fs.ls("/Volumes/claimsprocessing/bronze/member"))


path,name,size,modificationTime
dbfs:/Volumes/claimsprocessing/bronze/member/_SUCCESS,_SUCCESS,0,1785055479000
dbfs:/Volumes/claimsprocessing/bronze/member/_committed_7733815598757461083,_committed_7733815598757461083,124,1785055479000
dbfs:/Volumes/claimsprocessing/bronze/member/_started_7733815598757461083,_started_7733815598757461083,0,1785055478000
dbfs:/Volumes/claimsprocessing/bronze/member/part-00000-tid-7733815598757461083-d63edd58-d0d4-4aec-9624-60f34a72e1f7-425-1.c000.snappy.parquet,part-00000-tid-7733815598757461083-d63edd58-d0d4-4aec-9624-60f34a72e1f7-425-1.c000.snappy.parquet,10823,1785055479000
